# modeling_historical_risk_patch

Notebook hasil konversi dari `modeling_historical_risk_patch.py`. Kode dipisah ke beberapa cell berdasarkan blok top-level agar lebih mudah dibaca, dijalankan, dan di-screenshot.


## Imports


In [ ]:
from __future__ import annotations

import argparse
import contextlib
import io
import json


## Function: `feature_stack_channels`


In [ ]:
def feature_stack_channels(feature_stack: str) -> int:
    return 3 if feature_stack == "mask_context3" else 1


## Function: `build_parser`


In [ ]:
def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description=(
            "Ringkasan fase Modeling untuk historical risk patch. "
            "Script ini fokus pada arsitektur model, loss, optimizer, dan metrics tanpa menjalankan training."
        )
    )
    parser.add_argument("--seq-length", type=int, default=7)
    parser.add_argument("--patch-width", type=int, default=160)
    parser.add_argument("--patch-height", type=int, default=160)
    parser.add_argument("--feature-stack", choices=["mask", "mask_context3"], default="mask")
    parser.add_argument("--loss-strategy", choices=["wbce_dice", "wbce_dice_context"], default="wbce_dice_context")
    parser.add_argument("--context-kernel", type=int, default=5)
    parser.add_argument("--pos-weight", type=float, default=50.0)
    parser.add_argument(
        "--skip-tensorflow",
        action="store_true",
        help="Jangan instantiate model TensorFlow. Cocok jika hanya ingin ringkasan statis untuk screenshot kode.",
    )
    parser.add_argument(
        "--save-summary",
        default="",
        help="Path file txt untuk menyimpan model.summary(). Diabaikan jika TensorFlow tidak tersedia.",
    )
    parser.add_argument("--json", action="store_true")
    return parser


## Function: `build_static_layer_specs`


In [ ]:
def build_static_layer_specs(channels: int, seq_length: int, patch_height: int, patch_width: int) -> list[dict]:
    return [
        {
            "name": "Input",
            "type": "Input",
            "shape": [seq_length, patch_height, patch_width, channels],
        },
        {
            "name": "ConvLSTM Block 1",
            "type": "ConvLSTM2D",
            "filters": 32,
            "kernel_size": [3, 3],
            "padding": "same",
            "return_sequences": True,
        },
        {
            "name": "Normalization 1",
            "type": "BatchNormalization",
        },
        {
            "name": "Dropout 1",
            "type": "Dropout",
            "rate": 0.2,
        },
        {
            "name": "ConvLSTM Block 2",
            "type": "ConvLSTM2D",
            "filters": 32,
            "kernel_size": [3, 3],
            "padding": "same",
            "return_sequences": False,
        },
        {
            "name": "Normalization 2",
            "type": "BatchNormalization",
        },
        {
            "name": "Dropout 2",
            "type": "Dropout",
            "rate": 0.2,
        },
        {
            "name": "Conv Head",
            "type": "Conv2D",
            "filters": 16,
            "kernel_size": [3, 3],
            "activation": "relu",
            "padding": "same",
        },
        {
            "name": "Output Head",
            "type": "Conv2D",
            "filters": 1,
            "kernel_size": [1, 1],
            "activation": "sigmoid",
            "padding": "same",
        },
    ]


## Function: `import_tensorflow`


In [ ]:
def import_tensorflow():
    try:
        import tensorflow as tf  # type: ignore

        return tf, None
    except Exception as exc:
        return None, str(exc)


## Function: `_compute_weighted_bce`


In [ ]:
def _compute_weighted_bce(y_true, y_pred, pos_weight_tensor, tf):
    return tf.reduce_mean(
        -(
            (pos_weight_tensor * y_true * tf.math.log(y_pred))
            + ((1.0 - y_true) * tf.math.log(1.0 - y_pred))
        )
    )


## Function: `_compute_dice_loss`


In [ ]:
def _compute_dice_loss(y_true, y_pred, tf):
    intersection = tf.reduce_sum(y_true * y_pred)
    denominator = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return 1.0 - ((2.0 * intersection + 1.0) / (denominator + 1.0))


## Function: `_compute_context_bce`


In [ ]:
def _compute_context_bce(y_true, y_pred, context_kernel: int, tf):
    pooled_context = tf.nn.max_pool2d(y_true, ksize=context_kernel, strides=1, padding="SAME")
    context_weight = 1.0 + (2.0 * pooled_context)
    context_bce_map = -(
        (y_true * tf.math.log(y_pred))
        + ((1.0 - y_true) * tf.math.log(1.0 - y_pred))
    )
    return tf.reduce_mean(context_weight * context_bce_map)


## Function: `make_weighted_bce_dice_loss`


In [ ]:
def make_weighted_bce_dice_loss(pos_weight: float, tf):
    pos_weight_tensor = tf.constant(pos_weight, dtype=tf.float32)
    epsilon = tf.constant(1e-7, dtype=tf.float32)

    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), epsilon, 1.0 - epsilon)
        weighted_bce = _compute_weighted_bce(y_true, y_pred, pos_weight_tensor, tf)
        dice_loss = _compute_dice_loss(y_true, y_pred, tf)
        return (0.7 * weighted_bce) + (0.3 * dice_loss)

    return loss


## Function: `make_weighted_bce_dice_context_loss`


In [ ]:
def make_weighted_bce_dice_context_loss(pos_weight: float, context_kernel: int, tf):
    pos_weight_tensor = tf.constant(pos_weight, dtype=tf.float32)
    epsilon = tf.constant(1e-7, dtype=tf.float32)

    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), epsilon, 1.0 - epsilon)
        weighted_bce = _compute_weighted_bce(y_true, y_pred, pos_weight_tensor, tf)
        dice_loss = _compute_dice_loss(y_true, y_pred, tf)
        context_bce = _compute_context_bce(y_true, y_pred, context_kernel, tf)
        base_loss = (0.7 * weighted_bce) + (0.3 * dice_loss)
        return (0.75 * base_loss) + (0.25 * context_bce)

    return loss


## Function: `build_loss`


In [ ]:
def build_loss(loss_strategy: str, pos_weight: float, context_kernel: int, tf):
    if loss_strategy == "wbce_dice_context":
        return make_weighted_bce_dice_context_loss(pos_weight, context_kernel, tf)
    return make_weighted_bce_dice_loss(pos_weight, tf)


## Function: `build_model`


In [ ]:
def build_model(
    *,
    seq_length: int,
    patch_height: int,
    patch_width: int,
    channels: int,
    pos_weight: float,
    loss_strategy: str,
    context_kernel: int,
    tf,
):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(seq_length, patch_height, patch_width, channels)),
            tf.keras.layers.ConvLSTM2D(filters=32, kernel_size=(3, 3), padding="same", return_sequences=True),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.ConvLSTM2D(filters=32, kernel_size=(3, 3), padding="same", return_sequences=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Conv2D(filters=16, kernel_size=(3, 3), activation="relu", padding="same"),
            tf.keras.layers.Conv2D(filters=1, kernel_size=(1, 1), activation="sigmoid", padding="same"),
        ]
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=build_loss(loss_strategy, pos_weight, context_kernel, tf),
        jit_compile=False,
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.5),
            tf.keras.metrics.Precision(name="precision", thresholds=0.5),
            tf.keras.metrics.Recall(name="recall", thresholds=0.5),
            tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
        ],
    )
    return model


## Function: `capture_model_summary`


In [ ]:
def capture_model_summary(model) -> str:
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        model.summary()
    return buffer.getvalue().strip()


## Function: `build_summary`


In [ ]:
def build_summary(args: argparse.Namespace) -> dict:
    channels = feature_stack_channels(args.feature_stack)
    summary: dict = {
        "phase": "modeling",
        "model_profile": "historical_risk_patch_160",
        "seq_length": args.seq_length,
        "patch_width": args.patch_width,
        "patch_height": args.patch_height,
        "feature_stack": args.feature_stack,
        "channels": channels,
        "loss_strategy": args.loss_strategy,
        "context_kernel": args.context_kernel,
        "positive_class_weight": args.pos_weight,
        "optimizer": {"name": "Adam", "learning_rate": 0.001},
        "metrics": ["accuracy", "precision", "recall", "pr_auc"],
        "static_layers": build_static_layer_specs(
            channels=channels,
            seq_length=args.seq_length,
            patch_height=args.patch_height,
            patch_width=args.patch_width,
        ),
        "tensorflow_available": False,
        "tensorflow_error": None,
        "model_instantiated": False,
        "parameter_count": None,
        "model_summary_text": None,
    }

    if args.skip_tensorflow:
        summary["tensorflow_error"] = "skip_tensorflow=True"
        return summary

    tf, tf_error = import_tensorflow()
    if tf is None:
        summary["tensorflow_error"] = tf_error
        return summary

    summary["tensorflow_available"] = True
    model = build_model(
        seq_length=args.seq_length,
        patch_height=args.patch_height,
        patch_width=args.patch_width,
        channels=channels,
        pos_weight=args.pos_weight,
        loss_strategy=args.loss_strategy,
        context_kernel=args.context_kernel,
        tf=tf,
    )
    summary["model_instantiated"] = True
    summary["parameter_count"] = int(model.count_params())
    summary["model_summary_text"] = capture_model_summary(model)

    if args.save_summary:
        with open(args.save_summary, "w", encoding="utf-8") as handle:
            handle.write(str(summary["model_summary_text"]))
            handle.write("\n")
        summary["model_summary_saved_to"] = args.save_summary

    return summary


## Function: `print_human_summary`


In [ ]:
def print_human_summary(summary: dict) -> None:
    print("[modeling] Profile model:", summary["model_profile"])
    print("[modeling] Seq length:", summary["seq_length"])
    print("[modeling] Patch size:", f"{summary['patch_width']}x{summary['patch_height']}")
    print("[modeling] Feature stack:", summary["feature_stack"])
    print("[modeling] Channels:", summary["channels"])
    print("[modeling] Loss strategy:", summary["loss_strategy"])
    print("[modeling] Context kernel:", summary["context_kernel"])
    print("[modeling] Positive class weight:", f"{summary['positive_class_weight']:.4f}")
    print("[modeling] Optimizer:", f"{summary['optimizer']['name']} lr={summary['optimizer']['learning_rate']}")
    print("[modeling] Metrics:", ", ".join(summary["metrics"]))
    print("[modeling] Layer utama:")
    for layer in summary["static_layers"]:
        print(f"- {layer['name']} | {layer['type']}")

    if summary["tensorflow_available"] and summary["model_instantiated"]:
        print("[modeling] TensorFlow: tersedia")
        print("[modeling] Parameter model:", summary["parameter_count"])
        print()
        print(summary["model_summary_text"])
    else:
        print("[modeling] TensorFlow: tidak tersedia atau dilewati")
        print("[modeling] Detail:", summary["tensorflow_error"])


## Function: `main`


In [ ]:
def main() -> None:
    parser = build_parser()
    args = parser.parse_args()
    summary = build_summary(args)
    print_human_summary(summary)
    if args.json:
        print()
        print(json.dumps(summary, indent=2, ensure_ascii=False))


## Entry Point


In [ ]:
if __name__ == "__main__":
    main()
